In [1]:
!pip install folium


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import folium
from IPython.display import display

In [3]:
df = pd.read_csv("delhi_ncr_aqi_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip()

print(df.head())

              datetime        date  year  month  day  hour day_of_week  \
0  2020-01-01 06:00:00  2020-01-01  2020      1    1     6   Wednesday   
1  2020-01-01 12:00:00  2020-01-01  2020      1    1    12   Wednesday   
2  2020-01-01 18:00:00  2020-01-01  2020      1    1    18   Wednesday   
3  2020-01-01 23:00:00  2020-01-01  2020      1    1    23   Wednesday   
4  2020-01-01 06:00:00  2020-01-01  2020      1    1     6   Wednesday   

   is_weekend  season   city  ...    no2   so2    co    o3  temperature  \
0           0  winter  Delhi  ...  119.6  47.7  5.19  12.3          9.4   
1           0  winter  Delhi  ...  117.9  39.3  4.32  15.8         20.6   
2           0  winter  Delhi  ...  150.1  36.3  7.13  14.3         12.4   
3           0  winter  Delhi  ...  142.0  30.3  4.90  13.2         14.4   
4           0  winter  Delhi  ...  138.4  41.5  7.56  15.4          6.8   

   humidity  wind_speed  visibility  aqi  aqi_category  
0       100         3.6         1.2  500       

In [4]:
print(df.columns)

Index(['datetime', 'date', 'year', 'month', 'day', 'hour', 'day_of_week',
       'is_weekend', 'season', 'city', 'station', 'latitude', 'longitude',
       'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity',
       'wind_speed', 'visibility', 'aqi', 'aqi_category'],
      dtype='str')


In [5]:
sample_df = df.sample(200)

In [6]:
map_delhi = folium.Map(location=[28.61, 77.20], zoom_start=10)

In [7]:
for i in range(len(sample_df)):
    
    aqi = sample_df.iloc[i]['aqi']
    
    # Color logic
    if aqi <= 100:
        color = 'green'
    elif aqi <= 200:
        color = 'orange'
    else:
        color = 'red'
    
    folium.CircleMarker(
        location=[sample_df.iloc[i]['latitude'], sample_df.iloc[i]['longitude']],
        radius=6,
        popup=f"AQI: {aqi}",
        color=color,
        fill=True,
        fill_color=color
    ).add_to(map_delhi)

In [9]:
display(map_delhi)
map_delhi.save("delhi_pollution_map.html")
print("Map saved successfully!")

Map saved successfully!


In [1]:
import pandas as pd
import folium

summary = pd.read_csv("station_summary.csv")

map_delhi = folium.Map(
    location=[28.61, 77.20],
    zoom_start=10
)

for _, row in summary.iterrows():

    risk = row["risk_level"]

    if risk == "Critical":
        color = "red"

    elif risk == "High":
        color = "orange"

    elif risk == "Moderate":
        color = "blue"

    else:
        color = "green"

    popup_text = f"""
    <b>{row['station']}</b><br>

    AQI: {round(row['aqi'],1)}<br>

    Risk Level: {row['risk_level']}<br>

    Schools: {int(row['school_count'])}<br>

    Hospitals: {int(row['hospital_count'])}<br>

    Parks: {int(row['park_count'])}
    """

    folium.CircleMarker(

        location=[
            row["latitude"],
            row["longitude"]
        ],

        radius=row["aqi"]/40,

        popup=folium.Popup(
            popup_text,
            max_width=300
        ),

        color=color,

        fill=True,

        fill_color=color,

        fill_opacity=0.8

    ).add_to(map_delhi)

map_delhi.save(
    "delhi_pollution_map.html"
)

print("Map Updated Successfully")

Map Updated Successfully


In [6]:
import pandas as pd
import folium

# =====================================
# LOAD DATA
# =====================================

df = pd.read_csv("station_summary.csv")

print(df.head())

# =====================================
# CREATE MAP
# =====================================

map_delhi = folium.Map(
    location=[28.61, 77.20],
    zoom_start=10,
    tiles="CartoDB Positron"
)

# =====================================
# AQI COLOR FUNCTION
# =====================================

def get_color(aqi):

    if aqi >= 300:
        return "darkred"

    elif aqi >= 275:
        return "red"

    elif aqi >= 250:
        return "orange"

    else:
        return "yellow"

# =====================================
# ADD STATIONS
# =====================================

for _, row in df.iterrows():

    popup_html = f"""
    <h4>{row['station']}</h4>

    <b>🌫 AQI:</b> {round(row['aqi'],1)}<br>

    <b>⚠ Risk Level:</b> {row['risk_level']}<br>

    <hr>

    <b>🏫 Schools:</b> {int(row['school_count'])}<br>

    <b>🏥 Hospitals:</b> {int(row['hospital_count'])}<br>

    <b>🏞 Parks:</b> {int(row['park_count'])}<br>

    <b>📮 Pincode:</b> {row['pincode']}
    """

    folium.CircleMarker(

        location=[
            row["latitude"],
            row["longitude"]
        ],

        radius=max(
            8,
            row["aqi"] / 35
        ),

        color=get_color(
            row["aqi"]
        ),

        fill=True,

        fill_color=get_color(
            row["aqi"]
        ),

        fill_opacity=0.9,

        tooltip=f"""
        {row['station']}
        <br>
        AQI : {round(row['aqi'],1)}
        """,

        popup=folium.Popup(
            popup_html,
            max_width=300
        )

    ).add_to(map_delhi)

# =====================================
# AUTO FIT ALL STATIONS
# =====================================

bounds = [

    [
        df["latitude"].min(),
        df["longitude"].min()
    ],

    [
        df["latitude"].max(),
        df["longitude"].max()
    ]
]

map_delhi.fit_bounds(bounds)

# =====================================
# LEGEND
# =====================================

legend_html = """

<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 220px;
height: 180px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:15px;
padding:10px;
">

<b>AQI Legend</b>
<br><br>

🔴 AQI ≥ 300 Severe
<br>

🟠 AQI 275-299 Very Poor
<br>

🟡 AQI 250-274 Poor
<br>

🟢 AQI < 250 Moderate
<br><br>

Click Marker For Details

</div>

"""

map_delhi.get_root().html.add_child(
    folium.Element(legend_html)
)

# =====================================
# SAVE MAP
# =====================================

map_delhi.save(
    "delhi_pollution_map.html"
)

print(
    "Smart Pollution Map Created Successfully!"
)

               station         aqi  school_count  hospital_count  park_count  \
0   Anand Vihar, Delhi  295.166857          24.0            21.0       148.0   
1        Bawana, Delhi  283.694685           8.0             0.0         3.0   
2  Dwarka Sec 8, Delhi  248.200274           5.0             3.0        33.0   
3   Faridabad New Town  260.427464           0.0             0.0         0.0   
4    Faridabad Sec 16A  266.105497           0.0             0.0         0.0   

   latitude  longitude  pincode risk_level  
0   28.6469    77.3164   110092   Critical  
1   28.7762    77.0510   110039       High  
2   28.5708    77.0712   110077        Low  
3   28.3722    77.3099   121001   Moderate  
4   28.4089    77.3178   121001   Moderate  
Smart Pollution Map Created Successfully!
